### Instance methods

In [ ]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):    # <-- this is an instance method
        self.balance += amount

deposit is an instance method: any method whose first parameter is self. That's the entire definition. Every method you've written so far — __init__, deposit — has been an instance method. It's called "instance" because it operates on one specific instance's data (self.balance), and it needs an actual instance to be called on (acc1.deposit(50)) — you can't meaningfully call it without one, since there'd be no self to bind.

#### Concept 2: @classmethod

In [ ]:
class Account:
    bank_name = "SBI"

    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    @classmethod
    def get_bank_name(cls):
        print("  cls is:", cls)
        return cls.bank_name

acc1 = Account("Kavya", 100)

print("Result:", Account.get_bank_name())
print("Result:", acc1.get_bank_name())

### instance method → first parameter is self, receives whichever instance called it. Classmethod → first parameter is cls, always receives the class, regardless of how you call it.

In [ ]:
class Config:
    default_timeout = 30

    @classmethod
    def get_timeout(cls):
        return cls.default_timeout

    def get_timeout_via_self(self):
        return self.default_timeout

print("Config.get_timeout():", Config.get_timeout())
print(Config.get_timeout_via_self())

@classmethod lets you run class-level logic without requiring any object to exist first.

### @staticmethod


In [ ]:
class Demo:
    def instance_version(self, amount):
        print("instance_version received:", (self, amount))

    @staticmethod
    def static_version(amount):
        print("static_version received:", (amount,))

d = Demo()
d.instance_version(50)
d.static_version(50)

#### Instance method → instance (self) auto-injected 
#### Classmethod → class (cls) auto-injected
#### Staticmethod → nothing auto-injected — it's just a regular function that happens to be organized inside the class's namespace, callable via either

## All three, side by side

In [ ]:
acc1 = Account("Kavya", 100)

acc1.deposit(50)
print("acc1.deposit(50) -> acc1.balance =", acc1.balance)

print("Account.get_bank_name():", Account.get_bank_name())
print("acc1.get_bank_name()   :", acc1.get_bank_name())

print("Account.is_valid_amount(50):", Account.is_valid_amount(50))
print("acc1.is_valid_amount(50)   :", acc1.is_valid_amount(50))

instance methods are the only ones that truly need an instance to make sense — that's their whole point, operating on one account's private data. Classmethods and staticmethods don't care whether you reach them through the class or through some instance — they'll behave identically either way, since neither one is touching any particular instance's data.

#### Alternate constructors — the real reason @classmethod matters

In [5]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance

    @classmethod
    def from_config(cls, config):
        return cls(config["owner"], config["balance"])

class SavingsAccount(Account):
    pass

config = {"owner": "Kavya", "balance": 500}

acc1 = Account.from_config(config)
s1 = SavingsAccount.from_config(config)

print(type(acc1))
print(type(s1))

<class '__main__.Account'>
<class '__main__.SavingsAccount'>


#### SavingsAccount.from_config(config) correctly returned a SavingsAccount, not an Account — even though from_config is only written once, on the parent class.

That only worked because the method uses cls(...), not Account(...), to build the object. Remember — cls is always "whichever class you called this through." Call it via Account → cls is Account. Call it via SavingsAccount → cls is SavingsAccount. So cls(config["owner"], config["balance"]) automatically builds the correct type, without the method needing to know in advance which subclasses will ever exist. A @staticmethod couldn't do this — it has no way to know which class it was called through at all.

SavingsAccount.from_config_static(config) -> type: <class '__main__.Account'>
SavingsAccount.from_config_cls(config)    -> type: <class '__main__.SavingsAccount'>

SavingsAccount.from_config_static(config) ran without any error — you're right, it was inherited fine. But look at the type it produced: Account. Wrong. We called it through SavingsAccount, but got back a plain Account. That's because the staticmethod's body has Account(...) typed directly into it — a hardcoded name. No matter which class you call a staticmethod through, its body still says exactly what it says, word for word. It has no idea "oh, I was called via SavingsAccount this time" — that information never reaches it, because staticmethods take no automatic parameter at all.

The classmethod version got it right — SavingsAccount, correctly — because cls(...) isn't a hardcoded name. cls is "whichever class this call came through," filled in automatically, every single time, exactly like self gets filled in with whichever instance called a normal method.

In [6]:
class SavingsAccount(Account):
    pass

class HighYieldSavingsAccount(SavingsAccount):   # a grandchild, two levels deep
    pass

s = HighYieldSavingsAccount.from_config(config)
print(type(s))



<class '__main__.HighYieldSavingsAccount'>


#### When @staticmethod actually earns its keep

In [7]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        if not self.is_valid_amount(amount):
            print(f"  Rejected: {amount} is not a valid deposit amount")
            return
        self.balance += amount

    @staticmethod
    def is_valid_amount(amount):
        return amount > 0

acc1 = Account("Kavya", 100)
acc1.deposit(-50)
acc1.deposit(50)
print("Final balance:", acc1.balance)

  Rejected: -50 is not a valid deposit amount
Final balance: 150




self.is_valid_amount — Python checks self.__dict__ first. Not found (proven above).
Python then automatically checks self.__class__.__dict__ — which is Account.__dict__, because self.__class__ is Account. Found is_valid_amount there.